## 1) Locate WSI bag files (slide-level inputs)

We start WSI preprocessing from the precomputed bag files stored in:

`/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/bags/`

Each `*.h5` file corresponds to **one whole-slide image (WSI)** and represents a *bag of instances* (many tiles/patches extracted from the slide).

In this step, we:
- Scan the bags directory and collect all `.h5` files
- Extract a `slide_id` from each filename (e.g., `TCGA-4B-A93V`)
- Derive `patient_id` from the slide barcode to enable alignment with other modalities


**Output:**  
- A table mapping `bag_path → slide_id → patient_id`

In [1]:
import re
from pathlib import Path
import pandas as pd

BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
BAGS_DIR = BASE / "Data" / "bags"

print("BAGS_DIR:", BAGS_DIR)
print("Exists?:", BAGS_DIR.exists())

# Find all .h5 files (recursive, just in case there are subfolders)
bag_paths = sorted(BAGS_DIR.rglob("*.h5"))

print(f"\nTotal .h5 bag files found: {len(bag_paths)}")
if len(bag_paths) == 0:
    raise FileNotFoundError(f"No .h5 files found under: {BAGS_DIR}")

# Extract TCGA slide_id from filename (e.g., TCGA-4B-A93V.h5)
tcga_pattern = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", re.IGNORECASE)

rows = []
for p in bag_paths:
    m = tcga_pattern.search(p.name)
    if not m:
        # Keep it anyway, but mark slide_id as None
        slide_id = None
        patient_id = None
    else:
        slide_id = m.group(1).upper()
        # For TCGA, patient_id is first 12 chars: TCGA-XX-YYYY
        patient_id = slide_id[:12]
    rows.append({
        "bag_path": str(p),
        "filename": p.name,
        "slide_id": slide_id,
        "patient_id": patient_id
    })

bags_df = pd.DataFrame(rows)

print("\nPreview (first 10):")
display(bags_df.head(10))

# Basic QC
print("\nQC checks:")
print("Bags with missing slide_id:", bags_df["slide_id"].isna().sum())
print("Unique slide_ids:", bags_df["slide_id"].nunique(dropna=True))
print("Unique patient_ids:", bags_df["patient_id"].nunique(dropna=True))

# Show a few examples explicitly
print("\nExample slide_ids:")
print(bags_df["slide_id"].dropna().unique()[:10])

BAGS_DIR: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/bags
Exists?: True

Total .h5 bag files found: 830

Preview (first 10):


,bag_path,filename,slide_id,patient_id
0,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4384.h5,TCGA-05-4384,TCGA-05-4384
1,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4390.h5,TCGA-05-4390,TCGA-05-4390
2,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4396.h5,TCGA-05-4396,TCGA-05-4396
3,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4405.h5,TCGA-05-4405,TCGA-05-4405
4,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4410.h5,TCGA-05-4410,TCGA-05-4410
5,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4415.h5,TCGA-05-4415,TCGA-05-4415
6,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4417.h5,TCGA-05-4417,TCGA-05-4417
7,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4424.h5,TCGA-05-4424,TCGA-05-4424
8,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4425.h5,TCGA-05-4425,TCGA-05-4425
9,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4427.h5,TCGA-05-4427,TCGA-05-4427



QC checks:
Bags with missing slide_id: 0
Unique slide_ids: 830
Unique patient_ids: 830

Example slide_ids:
['TCGA-05-4384' 'TCGA-05-4390' 'TCGA-05-4396' 'TCGA-05-4405'
 'TCGA-05-4410' 'TCGA-05-4415' 'TCGA-05-4417' 'TCGA-05-4424'
 'TCGA-05-4425' 'TCGA-05-4427']


## 2) Decide the identifier used for alignment 

Whole Slide Image (WSI) data must be aligned with **RNA, DNA methylation, and clinical data** using a **true biological identifier**, not file UUIDs or arbitrary indices.

For TCGA data, the correct alignment key is the **TCGA barcode**.

From the WSI bag filenames:
- Each `.h5` filename corresponds to **one slide**
- The filename already contains the TCGA barcode

We define:
- `slide_id` = TCGA slide identifier (e.g., `TCGA-4B-A93V`)
- `patient_id` = first **12 characters** of the TCGA barcode  
  (e.g., `TCGA-4B-A93V`)

This ensures:
- Consistent patient-level alignment across modalities
- Correct handling of multiple tiles per slide
- No reliance on file UUIDs or directory structure

We only establish the **alignment key**.

**Output:**
- `slide_id`
- `patient_id` (used later for multimodal alignment)

In [ ]:
import h5py
from pathlib import Path

# Select ONE bag to inspect 
example_bag = Path(
    "/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/bags/TCGA-4B-A93V.h5"
)

print("Inspecting bag:", example_bag.name)
print("Exists?:", example_bag.exists())

with h5py.File(example_bag, "r") as f:
    print("\nTop-level keys in the .h5 file:")
    for key in f.keys():
        print(" -", key)

    print("\nDetailed inspection:")
    for key in f.keys():
        obj = f[key]
        if isinstance(obj, h5py.Dataset):
            print(f"Dataset: {key}")
            print("  shape:", obj.shape)
            print("  dtype:", obj.dtype)
        elif isinstance(obj, h5py.Group):
            print(f"Group: {key}")
            for subkey in obj.keys():
                subobj = obj[subkey]
                print(f"  - {subkey}: shape={getattr(subobj, 'shape', 'group')}")

print("\nInspection complete.")

Inspecting bag: TCGA-4B-A93V.h5
Exists?: True

Top-level keys in the .h5 file:
 - coords
 - features

Detailed inspection:
Dataset: coords
  shape: (2000, 2)
  dtype: int32
Dataset: features
  shape: (2000, 1024)
  dtype: float32

Inspection complete.


## 3) Confirm tile-level embeddings are already stored (and are consistent)

Your `.h5` bags already contain **tile embeddings**, so we do **not** need to run a pretrained encoder (UNI/ResNet/etc.) here.

From inspection:
- `coords`: tile locations on the slide
- `features`: tile embeddings (per-tile vectors)

Now we validate across the dataset that:
- Most bags contain both `coords` and `features`
- `features` have a consistent embedding size (e.g., 1024)
- We record basic QC like number of tiles per bag

**Output:** a QC summary table of bags (`n_tiles`, `embed_dim`, presence of keys), plus a quick consistency report.

In [3]:
import pandas as pd
import numpy as np
import h5py
from pathlib import Path

BAGS_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/bags")

bag_paths = sorted(BAGS_DIR.glob("*.h5"))
print("Total bags:", len(bag_paths))

rows = []
for p in bag_paths:
    slide_id = p.stem
    patient_id = slide_id[:12]

    try:
        with h5py.File(p, "r") as f:
            has_coords = "coords" in f
            has_feats  = "features" in f

            n_tiles = None
            embed_dim = None

            if has_feats:
                feats = f["features"]
                n_tiles = feats.shape[0]
                embed_dim = feats.shape[1] if len(feats.shape) == 2 else None

            rows.append({
                "slide_id": slide_id,
                "patient_id": patient_id,
                "bag_path": str(p),
                "has_coords": has_coords,
                "has_features": has_feats,
                "n_tiles": n_tiles,
                "embed_dim": embed_dim
            })

    except Exception as e:
        rows.append({
            "slide_id": slide_id,
            "patient_id": patient_id,
            "bag_path": str(p),
            "has_coords": False,
            "has_features": False,
            "n_tiles": None,
            "embed_dim": None,
            "error": str(e)
        })

bags_qc = pd.DataFrame(rows)

print("\nKey presence counts:")
print(bags_qc[["has_coords", "has_features"]].value_counts(dropna=False))

print("\nEmbedding dim distribution (top 10):")
print(bags_qc["embed_dim"].value_counts(dropna=False).head(10))

print("\nTile count summary (n_tiles):")
print(bags_qc["n_tiles"].describe())

print("\nPreview (first 10 rows):")
display(bags_qc.head(10))

# Optional: save QC table for reproducibility
OUT_QC = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi")
OUT_QC.mkdir(parents=True, exist_ok=True)

bags_qc.to_csv(OUT_QC / "wsi_bags_qc.csv", index=False)
print("\nSaved QC table to:", OUT_QC / "wsi_bags_qc.csv")


Total bags: 830

Key presence counts:
has_coords  has_features
True        True            829
False       False             1
Name: count, dtype: int64

Embedding dim distribution (top 10):
embed_dim
1024.0    829
NaN         1
Name: count, dtype: int64

Tile count summary (n_tiles):
count     829.000000
mean     1974.188179
std       195.239085
min        34.000000
25%      2000.000000
50%      2000.000000
75%      2000.000000
max      2000.000000
Name: n_tiles, dtype: float64

Preview (first 10 rows):


,slide_id,patient_id,bag_path,has_coords,has_features,n_tiles,embed_dim,error
0,TCGA-05-4384,TCGA-05-4384,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,79.0,1024.0,NaN
1,TCGA-05-4390,TCGA-05-4390,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,770.0,1024.0,NaN
2,TCGA-05-4396,TCGA-05-4396,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,2000.0,1024.0,NaN
3,TCGA-05-4405,TCGA-05-4405,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,2000.0,1024.0,NaN
4,TCGA-05-4410,TCGA-05-4410,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,454.0,1024.0,NaN
5,TCGA-05-4415,TCGA-05-4415,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,2000.0,1024.0,NaN
6,TCGA-05-4417,TCGA-05-4417,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,2000.0,1024.0,NaN
7,TCGA-05-4424,TCGA-05-4424,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,2000.0,1024.0,NaN
8,TCGA-05-4425,TCGA-05-4425,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,59.0,1024.0,NaN
9,TCGA-05-4427,TCGA-05-4427,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,True,True,2000.0,1024.0,NaN



Saved QC table to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_bags_qc.csv


## 4) Aggregate tile embeddings → slide-level embedding (MIL pooling)

Each WSI bag contains a **variable number of tile embeddings** (e.g., up to 2000 tiles per slide, each 1024-D).

Classical fusion models (early / late fusion) require a **fixed-length vector per sample**, so we must aggregate tile-level embeddings into **one slide-level embedding per bag**.

At this stage:
- We use **mean pooling** as a strong, simple MIL baseline
- No learning happens here (pure aggregation)
- One vector per slide is produced

Mean pooling:
- Computes the average across all tile embeddings in a bag
- Preserves global signal across the slide
- Common baseline in WSI MIL pipelines

**Output (per slide):**
- `slide_embedding` with shape `(embed_dim,)` (e.g., 1024)

**Output (dataset-level):**
- One slide-level embedding per bag
- Aligned to `slide_id` and `patient_id`

In [4]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

# Paths
BAGS_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/bags")
OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load previously built bag table (or rebuild quickly)
bag_files = sorted(BAGS_DIR.glob("*.h5"))

records = []

for bag_path in bag_files:
    slide_id = bag_path.stem
    patient_id = slide_id[:12]

    try:
        with h5py.File(bag_path, "r") as f:
            if "features" not in f:
                raise ValueError("Missing features dataset")

            feats = f["features"][:]  # (n_tiles, embed_dim)
            slide_embedding = feats.mean(axis=0)  # mean pooling

            records.append({
                "slide_id": slide_id,
                "patient_id": patient_id,
                "embedding": slide_embedding
            })

    except Exception as e:
        print(f"[ERROR] {slide_id}: {e}")

# Build DataFrame
wsi_df = pd.DataFrame(records)

print("Total aggregated slides:", len(wsi_df))
print("Embedding shape:", wsi_df.iloc[0]["embedding"].shape)

# Stack embeddings into matrix
X_wsi = np.stack(wsi_df["embedding"].values)
slide_ids = wsi_df["slide_id"].values
patient_ids = wsi_df["patient_id"].values

print("WSI matrix shape:", X_wsi.shape)

# Save outputs
np.save(OUT_DIR / "wsi_slide_embeddings.npy", X_wsi.astype(np.float32))
np.save(OUT_DIR / "wsi_slide_ids.npy", slide_ids)
np.save(OUT_DIR / "wsi_patient_ids.npy", patient_ids)

print("Saved:")
print(" - wsi_slide_embeddings.npy")
print(" - wsi_slide_ids.npy")
print(" - wsi_patient_ids.npy")


[ERROR] TCGA-75-6205: Unable to synchronously open file (bad object header version number)
Total aggregated slides: 829
Embedding shape: (1024,)
WSI matrix shape: (829, 1024)
Saved:
 - wsi_slide_embeddings.npy
 - wsi_slide_ids.npy
 - wsi_patient_ids.npy


## 5) Build the WSI feature matrix across all slides

Now that each bag has been reduced to **one fixed-length slide embedding** (via mean pooling), we stack all slide embeddings into one matrix.

This gives us the WSI feature matrix:

- `X_wsi`: shape `(N_slides, embed_dim)` → here `embed_dim = 1024`
- `wsi_slide_ids`: length `N_slides`
- `wsi_patient_ids`: length `N_slides`

Notes:
- `N_slides` may be slightly less than total `.h5` files if any bags are corrupted or unreadable.
- This step is still **before alignment** to the RNA split.

**Output:**
- `X_wsi`
- `wsi_slide_ids`
- `wsi_patient_ids`
- Saved numpy artifacts for downstream alignment

In [5]:
import numpy as np
from pathlib import Path

OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi")

X_wsi = np.load(OUT_DIR / "wsi_slide_embeddings.npy")
wsi_slide_ids = np.load(OUT_DIR / "wsi_slide_ids.npy", allow_pickle=True)
wsi_patient_ids = np.load(OUT_DIR / "wsi_patient_ids.npy", allow_pickle=True)

print("Loaded WSI artifacts:")
print("X_wsi shape:", X_wsi.shape)
print("wsi_slide_ids:", wsi_slide_ids.shape, "| example:", wsi_slide_ids[:5])
print("wsi_patient_ids:", wsi_patient_ids.shape, "| example:", wsi_patient_ids[:5])

# Basic QC
print("\nQC checks:")
print("Any NaNs in X_wsi?:", np.isnan(X_wsi).any())
print("Unique slide IDs:", len(np.unique(wsi_slide_ids)))
print("Unique patient IDs:", len(np.unique(wsi_patient_ids)))

# Optional: show how many patients have multiple slides
_, counts = np.unique(wsi_patient_ids, return_counts=True)
print("Patients with >1 slide:", int((counts > 1).sum()))
print("Max slides per patient:", int(counts.max()))


Loaded WSI artifacts:
X_wsi shape: (829, 1024)
wsi_slide_ids: (829,) | example: ['TCGA-05-4384' 'TCGA-05-4390' 'TCGA-05-4396' 'TCGA-05-4405'
 'TCGA-05-4410']
wsi_patient_ids: (829,) | example: ['TCGA-05-4384' 'TCGA-05-4390' 'TCGA-05-4396' 'TCGA-05-4405'
 'TCGA-05-4410']

QC checks:
Any NaNs in X_wsi?: False
Unique slide IDs: 829
Unique patient IDs: 829
Patients with >1 slide: 0
Max slides per patient: 1


## 6) Align WSI to the RNA train/test split (leakage-safe)

Now we align WSI **to the exact same patients and order** used in the RNA split.

Why this matters:
- We **never re-split** WSI independently (that would break multimodal pairing and can leak information).
- We use RNA’s `train_patient_ids` and `test_patient_ids` as the **source of truth**.

What we do:
- Load:
  - `rna_train_patient_ids.npy` (660)
  - `rna_test_patient_ids.npy` (165)
- Build a lookup from `patient_id → WSI embedding`
- Subset WSI to only those patients
- Force the row alignment order to match RNA **exactly**

**Output:**
- `X_wsi_train` (expected `(660, 1024)` if all train patients have WSI)
- `X_wsi_test`  (expected `(165, 1024)` if all test patients have WSI)
- `wsi_train_patient_ids`, `wsi_test_patient_ids`

Saved:
- `wsi_train.npy`, `wsi_test.npy`
- `wsi_train_patient_ids.npy`, `wsi_test_patient_ids.npy`
- `wsi_missing_train.npy`, `wsi_missing_test.npy` (for QC)

In [6]:
import numpy as np
from pathlib import Path

BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
PREP_DIR = BASE / "Data" / "02_preprocessed"

WSI_DIR = PREP_DIR / "wsi"
RNA_DIR = PREP_DIR / "rna"

# Load WSI artifacts (already built)
X_wsi = np.load(WSI_DIR / "wsi_slide_embeddings.npy")  # (829, 1024)
wsi_patient_ids = np.load(WSI_DIR / "wsi_patient_ids.npy", allow_pickle=True).astype(str)

# Load RNA split patient IDs (source of truth)
rna_train_ids = np.load(RNA_DIR / "rna_train_patient_ids.npy", allow_pickle=True).astype(str)
rna_test_ids  = np.load(RNA_DIR / "rna_test_patient_ids.npy", allow_pickle=True).astype(str)

print("RNA split sizes:", len(rna_train_ids), len(rna_test_ids))
print("WSI total:", X_wsi.shape, "| unique patients:", len(np.unique(wsi_patient_ids)))

# Build patient_id -> row index lookup for WSI
wsi_index = {pid: i for i, pid in enumerate(wsi_patient_ids)}

def align_wsi_to_ids(target_ids, X, index_map):
    missing = []
    rows = []
    for pid in target_ids:
        if pid in index_map:
            rows.append(X[index_map[pid]])
        else:
            missing.append(pid)
    X_aligned = np.stack(rows, axis=0) if len(rows) > 0 else np.zeros((0, X.shape[1]), dtype=X.dtype)
    return X_aligned, np.array(missing, dtype=object)

# Align train/test
X_wsi_train, wsi_missing_train = align_wsi_to_ids(rna_train_ids, X_wsi, wsi_index)
X_wsi_test,  wsi_missing_test  = align_wsi_to_ids(rna_test_ids,  X_wsi, wsi_index)

print("\nAligned WSI shapes:")
print("X_wsi_train:", X_wsi_train.shape)
print("X_wsi_test :", X_wsi_test.shape)

print("\nMissing WSI patients:")
print("Train missing:", len(wsi_missing_train))
print("Test  missing:", len(wsi_missing_test))

# Sanity checks
assert X_wsi_train.shape[0] == (len(rna_train_ids) - len(wsi_missing_train))
assert X_wsi_test.shape[0]  == (len(rna_test_ids)  - len(wsi_missing_test))
assert X_wsi_train.shape[1] == X_wsi.shape[1] == 1024
assert X_wsi_test.shape[1]  == X_wsi.shape[1] == 1024

# Save aligned artifacts
np.save(WSI_DIR / "wsi_train.npy", X_wsi_train.astype(np.float32))
np.save(WSI_DIR / "wsi_test.npy", X_wsi_test.astype(np.float32))

np.save(WSI_DIR / "wsi_train_patient_ids.npy", rna_train_ids)  # keep RNA order
np.save(WSI_DIR / "wsi_test_patient_ids.npy", rna_test_ids)

np.save(WSI_DIR / "wsi_missing_train.npy", wsi_missing_train)
np.save(WSI_DIR / "wsi_missing_test.npy", wsi_missing_test)

print("\nSaved:")
print(" -", WSI_DIR / "wsi_train.npy")
print(" -", WSI_DIR / "wsi_test.npy")
print(" -", WSI_DIR / "wsi_train_patient_ids.npy")
print(" -", WSI_DIR / "wsi_test_patient_ids.npy")
print(" -", WSI_DIR / "wsi_missing_train.npy")
print(" -", WSI_DIR / "wsi_missing_test.npy")


RNA split sizes: 660 165
WSI total: (829, 1024) | unique patients: 829

Aligned WSI shapes:
X_wsi_train: (658, 1024)
X_wsi_test : (165, 1024)

Missing WSI patients:
Train missing: 2
Test  missing: 0

Saved:
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_train.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_test.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_train_patient_ids.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_test_patient_ids.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_missing_train.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi/wsi_missing_test.npy


## 7) Define the final multimodal patient set 

WSI is missing for **2 train patients**, so for strict multimodal fusion we keep only patients that have:
- RNA (train/test split source of truth)
- WSI embeddings available

We create:
- `final_train_patient_ids` = RNA train patients **minus** missing WSI train patients
- `final_test_patient_ids`  = unchanged (since test has no missing WSI)

These IDs will be used to subset **clinical, methylation, and RNA** so that every modality has the exact same patients in the exact same order.

**Output:**
- `final_train_patient_ids.npy`
- `final_test_patient_ids.npy`

In [7]:
import numpy as np
from pathlib import Path

BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
PREP_DIR = BASE / "Data" / "02_preprocessed"
WSI_DIR = PREP_DIR / "wsi"
RNA_DIR = PREP_DIR / "rna"

rna_train_ids = np.load(RNA_DIR / "rna_train_patient_ids.npy", allow_pickle=True).astype(str)
rna_test_ids  = np.load(RNA_DIR / "rna_test_patient_ids.npy", allow_pickle=True).astype(str)

wsi_missing_train = np.load(WSI_DIR / "wsi_missing_train.npy", allow_pickle=True).astype(str)
wsi_missing_test  = np.load(WSI_DIR / "wsi_missing_test.npy", allow_pickle=True).astype(str)

missing_train_set = set(wsi_missing_train.tolist())
missing_test_set  = set(wsi_missing_test.tolist())

final_train_ids = np.array([pid for pid in rna_train_ids if pid not in missing_train_set], dtype=object)
final_test_ids  = np.array([pid for pid in rna_test_ids  if pid not in missing_test_set], dtype=object)

print("Final strict-fusion IDs:")
print("Train:", len(final_train_ids), "/", len(rna_train_ids), " (dropped:", len(rna_train_ids) - len(final_train_ids), ")")
print("Test :", len(final_test_ids), "/", len(rna_test_ids),  " (dropped:", len(rna_test_ids) - len(final_test_ids), ")")

np.save(PREP_DIR / "final_train_patient_ids.npy", final_train_ids)
np.save(PREP_DIR / "final_test_patient_ids.npy", final_test_ids)

print("\nSaved:")
print(" -", PREP_DIR / "final_train_patient_ids.npy")
print(" -", PREP_DIR / "final_test_patient_ids.npy")


Final strict-fusion IDs:
Train: 658 / 660  (dropped: 2 )
Test : 165 / 165  (dropped: 0 )

Saved:
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/final_train_patient_ids.npy
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/final_test_patient_ids.npy


## 7) Standardize WSI features (train-only)

To prevent data leakage:
- Fit the scaler **only on training WSI embeddings**
- Apply the same scaler to both train and test
- Save the scaled arrays and the scaler for reproducibility

**Outputs:**
- `wsi_train_scaled.npy`
- `wsi_test_scaled.npy`
- `wsi_scaler.joblib`

In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
from pathlib import Path

BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
WSI_DIR = BASE / "Data" / "02_preprocessed" / "wsi"

X_wsi_train = np.load(WSI_DIR / "wsi_train.npy")
X_wsi_test  = np.load(WSI_DIR / "wsi_test.npy")

scaler = StandardScaler()
X_wsi_train_scaled = scaler.fit_transform(X_wsi_train)
X_wsi_test_scaled  = scaler.transform(X_wsi_test)

np.save(WSI_DIR / "wsi_train_scaled.npy", X_wsi_train_scaled)
np.save(WSI_DIR / "wsi_test_scaled.npy", X_wsi_test_scaled)
joblib.dump(scaler, WSI_DIR / "wsi_scaler.joblib")

print("WSI standardization complete.")
print("Train shape:", X_wsi_train_scaled.shape)
print("Test  shape:", X_wsi_test_scaled.shape)


WSI standardization complete.
Train shape: (658, 1024)
Test  shape: (165, 1024)


## 8) Dimensionality reduction (WSI PCA)

WSI embeddings are high-dimensional (1024-d).
To stabilize fusion models and reduce noise:

- Fit PCA **on training data only**
- Keep components explaining **90% variance**
- Transform both train and test with the same PCA

**Outputs:**
- `wsi_train_pca.npy`
- `wsi_test_pca.npy`
- `wsi_pca.joblib`

In [9]:
import numpy as np
from sklearn.decomposition import PCA
import joblib
from pathlib import Path

BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
WSI_DIR = BASE / "Data" / "02_preprocessed" / "wsi"

X_wsi_train_scaled = np.load(WSI_DIR / "wsi_train_scaled.npy")
X_wsi_test_scaled  = np.load(WSI_DIR / "wsi_test_scaled.npy")

pca = PCA(n_components=0.90, random_state=42)
X_wsi_train_pca = pca.fit_transform(X_wsi_train_scaled)
X_wsi_test_pca  = pca.transform(X_wsi_test_scaled)

np.save(WSI_DIR / "wsi_train_pca.npy", X_wsi_train_pca)
np.save(WSI_DIR / "wsi_test_pca.npy", X_wsi_test_pca)
joblib.dump(pca, WSI_DIR / "wsi_pca.joblib")

print("WSI PCA complete.")
print("Original dim:", X_wsi_train_scaled.shape[1])
print("Reduced dim :", X_wsi_train_pca.shape[1])
print("Train PCA shape:", X_wsi_train_pca.shape)
print("Test  PCA shape:", X_wsi_test_pca.shape)


WSI PCA complete.
Original dim: 1024
Reduced dim : 110
Train PCA shape: (658, 110)
Test  PCA shape: (165, 110)
